In [6]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem import Descriptors
from tqdm import tqdm
import logging
import time
from func_timeout import func_timeout, FunctionTimedOut
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb

# Set up logging for errors during descriptor computation
logging.basicConfig(filename='descriptor_errors.log', level=logging.INFO,
                    format='%(asctime)s:%(levelname)s:%(message)s')

# -------------------------------
# 1. Load Training and Testing Data
# -------------------------------
train_file = "./dw_data/Opt1_acidic_tr.csv"
test_file = "./dw_data/Opt2_acidic_tst.csv"
smiles_col = 'OriginalSmiles'
target = 'pKa'

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

# Create a SaltRemover instance.
saltRemover = SaltRemover(defnFilename='./Salts.txt')

# Compute the RDKit Mol object (and strip salts) for both datasets.
for df in [train_df, test_df]:
    df['Mol'] = df[smiles_col].astype(str).apply(
        lambda s: saltRemover.StripMol(Chem.MolFromSmiles(s))
    )

print(f"Training: Total molecules: {len(train_df)}; Valid molecules: {train_df['Mol'].notnull().sum()}")
print(f"Testing: Total molecules: {len(test_df)}; Valid molecules: {test_df['Mol'].notnull().sum()}")

# -------------------------------
# 2. Descriptor Computation Functions
# -------------------------------
def safe_call(func, mol, timeout=1):
    """
    Calls a function with a timeout using func_timeout.
    Returns the result if completed within 'timeout' seconds,
    otherwise returns np.nan.
    """
    try:
        result = func_timeout(timeout, func, args=(mol,))
        return result
    except FunctionTimedOut:
        logging.info(f"Timeout in {func.__name__}")
        return np.nan
    except Exception as e:
        logging.info(f"Error in {func.__name__}: {e}")
        return np.nan

def compute_descriptors_for_mol(mol):
    """
    Computes all RDKit descriptors for a given molecule,
    excluding descriptors known to cause errors.
    Each descriptor call is wrapped in safe_call() with a timeout.
    """
    # Define descriptors to exclude
    error_descriptors = {
        "MaxEStateIndex", "MinEStateIndex", 
        "MaxAbsEStateIndex", "MinAbsEStateIndex",
        "FpDensityMorgan1", "FpDensityMorgan2", "FpDensityMorgan3"
    }
    filtered_desc_funcs = {name: func for name, func in Descriptors.descList 
                             if name not in error_descriptors}
    
    if mol is None or Chem.MolToSmiles(mol) == '':
        return None
    
    desc_values = {}
    for name, func in filtered_desc_funcs.items():
        value = safe_call(func, mol, timeout=1)
        desc_values[name] = value
    return desc_values

def compute_descriptors_for_df(df, dataset_label):
    """
    Compute descriptors for each molecule in the given dataframe.
    The computed descriptor dicts are then joined with the original data.
    A new column 'Dataset' is added to indicate 'train' or 'test'.
    """
    desc_list = []
    start_time = time.time()
    mols = df['Mol'].tolist()
    for i, mol in tqdm(enumerate(mols), total=len(mols),
                       desc=f"Computing descriptors for {dataset_label}"):
        try:
            result = compute_descriptors_for_mol(mol)
            # If a molecule fails descriptor computation, keep it as an empty dict
            desc_list.append(result if result is not None else {})
        except Exception as e:
            logging.info(f"Molecule {i} error: {e}")
            desc_list.append({})
    end_time = time.time()
    print(f"Descriptor computation for {dataset_label} took {end_time - start_time:.2f} seconds.")
    
    # Create a DataFrame of descriptors
    desc_df = pd.DataFrame(desc_list)
    # Optionally, drop rows that are completely empty (if any)
    desc_df = desc_df.dropna(how='all').reset_index(drop=True)
    # Remove descriptor columns with zero variance.
    desc_df = desc_df.loc[:, desc_df.std() > 0]
    
    # Append the descriptor values to the original data
    df_reset = df.reset_index(drop=True)
    combined_df = pd.concat([df_reset, desc_df], axis=1)
    combined_df['Dataset'] = dataset_label
    return combined_df, desc_df

# Compute descriptors for training and testing datasets.
train_combined, train_desc_df = compute_descriptors_for_df(train_df, "train")
test_combined, test_desc_df = compute_descriptors_for_df(test_df, "test")

# Save combined descriptors (for all compounds) to a CSV file.
combined_descriptors = pd.concat([train_combined, test_combined], axis=0).reset_index(drop=True)
combined_descriptors.to_csv("combined_descriptors.csv", index=False)
print("Combined descriptors (train and test) saved to combined_descriptors.csv")

# -------------------------------
# 3. Modeling Functions (Using External Test Set)
# -------------------------------
param_grid = {
    'n_estimators': [300],
    'max_depth': [3, 6],
    'learning_rate': [0.1, 0.2],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda': [1, 5]
}

def train_and_evaluate_single_descriptor(X_train, y_train, X_test, y_test):
    xgbr = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
    grid_search = GridSearchCV(estimator=xgbr, param_grid=param_grid,
                               scoring='r2', cv=5, n_jobs=5, verbose=0)
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    y_pred_test = best_model.predict(X_test)
    test_r2 = r2_score(y_test, y_pred_test)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_mse = mean_squared_error(y_test, y_pred_test)
    test_rmse = np.sqrt(test_mse)
    return {
        'Test_R2': test_r2,
        'Test_MAE': test_mae,
        'Test_MSE': test_mse,
        'Test_RMSE': test_rmse,
        'Best_Params': str(grid_search.best_params_)
    }

# -------------------------------
# 4. Evaluate Each Descriptor Individually (Using Training for fitting and External Testing)
# -------------------------------
performance_list = []
descriptor_columns = train_desc_df.columns  # these are the computed descriptor names

print("Evaluating each descriptor individually using the external test set...")
for col in tqdm(descriptor_columns, total=len(descriptor_columns)):
    try:
        # For each descriptor, drop rows with missing values in that column separately.
        train_single = train_combined.dropna(subset=[col]).reset_index(drop=True)
        test_single = test_combined.dropna(subset=[col]).reset_index(drop=True)
        
        # Ensure that there are enough samples in both sets.
        if train_single.shape[0] < 10 or test_single.shape[0] < 10:
            continue
        
        X_train_single = train_single[[col]].values.astype(float)
        y_train_single = train_single[target].values
        X_test_single = test_single[[col]].values.astype(float)
        y_test_single = test_single[target].values
        
        metrics = train_and_evaluate_single_descriptor(X_train_single, y_train_single,
                                                       X_test_single, y_test_single)
        metrics['Descriptor'] = col
        performance_list.append(metrics)
    except Exception as e:
        logging.info(f"Error evaluating descriptor {col}: {e}")

performance_df = pd.DataFrame(performance_list)
performance_df.sort_values(by='Test_R2', ascending=False, inplace=True)
performance_df.to_csv("individual_descriptor_performance.csv", index=False)
print("Individual descriptor performance saved to individual_descriptor_performance.csv")
print("\nBest individual descriptors based on Test R2:")
print(performance_df[['Descriptor', 'Test_R2']].head())

# -------------------------------
# 5. Combine the Best Descriptors and Train a Combined Model
# -------------------------------
# Select top N descriptors based on external test set performance.
num_best = 190
best_descriptors = performance_df.head(num_best)['Descriptor'].tolist()
print(f"\nCombining the top {num_best} descriptors: {best_descriptors}")

# For the combined model, drop rows in both training and testing data that have NaNs in any of the best descriptors.
train_model_df = train_combined.dropna(subset=best_descriptors).reset_index(drop=True)
test_model_df = test_combined.dropna(subset=best_descriptors).reset_index(drop=True)

X_train_combined = train_model_df[best_descriptors].values.astype(float)
y_train_combined = train_model_df[target].values
X_test_combined = test_model_df[best_descriptors].values.astype(float)
y_test_combined = test_model_df[target].values

# Train combined model with GridSearchCV
xgbr = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
grid_search = GridSearchCV(estimator=xgbr, param_grid=param_grid,
                           scoring='r2', cv=5, n_jobs=5, verbose=0)
grid_search.fit(X_train_combined, y_train_combined)
best_model = grid_search.best_estimator_
y_pred_train = best_model.predict(X_train_combined)
y_pred_test = best_model.predict(X_test_combined)

combined_metrics = {
    'Train_R2': r2_score(y_train_combined, y_pred_train),
    'Test_R2': r2_score(y_test_combined, y_pred_test),
    'Test_MAE': mean_absolute_error(y_test_combined, y_pred_test),
    'Test_MSE': mean_squared_error(y_test_combined, y_pred_test),
    'Test_RMSE': np.sqrt(mean_squared_error(y_test_combined, y_pred_test)),
    'Best_Params': str(grid_search.best_params_)
}

print("\nCombined model performance using top descriptors (external test set):")
for k, v in combined_metrics.items():
    print(f"{k}: {v}")

combined_df = pd.DataFrame([combined_metrics])
combined_df.to_csv("combined_descriptor_model_performance.csv", index=False)
print("Combined model performance saved to combined_descriptor_model_performance.csv")


Training: Total molecules: 2445; Valid molecules: 2445
Testing: Total molecules: 815; Valid molecules: 815


Computing descriptors for train: 100%|█████████████████████████████████████████████| 2445/2445 [01:22<00:00, 29.51it/s]


Descriptor computation for train took 82.86 seconds.


Computing descriptors for test: 100%|████████████████████████████████████████████████| 815/815 [00:27<00:00, 29.47it/s]


Descriptor computation for test took 27.66 seconds.
Combined descriptors (train and test) saved to combined_descriptors.csv
Evaluating each descriptor individually using the external test set...


100%|████████████████████████████████████████████████████████████████████████████████| 192/192 [04:47<00:00,  1.50s/it]


Individual descriptor performance saved to individual_descriptor_performance.csv

Best individual descriptors based on Test R2:
             Descriptor   Test_R2
8   MinAbsPartialCharge  0.331018
5      MaxPartialCharge  0.324624
7   MaxAbsPartialCharge  0.315330
6      MinPartialCharge  0.313271
51             SMR_VSA1  0.302956

Combining the top 190 descriptors: ['MinAbsPartialCharge', 'MaxPartialCharge', 'MaxAbsPartialCharge', 'MinPartialCharge', 'SMR_VSA1', 'TPSA', 'PEOE_VSA14', 'BCUT2D_MRLOW', 'EState_VSA1', 'fr_COO', 'fr_COO2', 'SlogP_VSA2', 'BCUT2D_MWHI', 'SMR_VSA10', 'fr_Al_COO', 'EState_VSA9', 'EState_VSA10', 'PEOE_VSA3', 'VSA_EState2', 'VSA_EState5', 'PEOE_VSA10', 'VSA_EState3', 'MolLogP', 'SMR_VSA7', 'fr_C_O', 'EState_VSA2', 'PEOE_VSA9', 'PEOE_VSA8', 'PEOE_VSA2', 'EState_VSA8', 'EState_VSA4', 'PEOE_VSA1', 'BCUT2D_MRHI', 'SMR_VSA3', 'SMR_VSA9', 'SlogP_VSA3', 'SMR_VSA6', 'fr_Ar_COO', 'NumHeteroatoms', 'PEOE_VSA11', 'PEOE_VSA12', 'EState_VSA7', 'ExactMolWt', 'BCUT2D_LOGPLOW', 

In [7]:
print("Hello")

Hello


In [8]:
print("wow")

wow
